In [1]:
import ollama

# AI TEACHER INSTRUCTIONS
TEACHER_INSTRUCTION = """
You are an expert AI Teacher.

Your job is to teach students, not just provide answers.

Teaching rules:
- Explain concepts using simple, everyday language.
- Keep every answer precise and limited to 3 lines.
- Adapt explanations to the student's level.
- Use the preferred language specified by the student.
- Focus on explaining the concept, not just giving the answer.
"""

# CONVERSATION MEMORY
conversation = [{"role": "system", "content": TEACHER_INSTRUCTION}]

# AI TEACHER FUNCTION
def ask_teacher(question, subject="AI & Data Science", student_level="Beginner", language="English"):
    
    user_message = f"""
Subject: {subject}
Student Level: {student_level}
Preferred Language: {language}
Student Question:
{question}"""

    conversation.append({"role": "user", "content": user_message})
    
    response = ollama.chat(model="llama3.2", messages=conversation)

    answer = response["message"]["content"]

    conversation.append({"role": "assistant", "content": answer})
    return answer

# CHAT APPLICATION 

def start_teacher():
    print("=" * 60)
    print("                AI TEACHER")
    print("               Powered by Llama 3.2")
    print("=" * 60)

    print("Type 'exit' to stop.\n")

    while True:
        question = input("Student: ").strip()

        if question.lower() == "exit":
            print("\nTeacher: Keep learning! 👋")
            break

        if not question:
            continue

        try:
            answer = ask_teacher(question=question, subject="AI & Data Science", student_level="Beginner", language="English")

            print("\nTeacher:")
            print(answer)

        except Exception as error:
            print(f"\nError: {error}")
            print("check that Ollama is running and Llama3.2 is installed.")
            
        print("\n" + "-" * 60)

# RUN APPLICATION

if __name__ == "__main__":
    start_teacher()

                AI TEACHER
               Powered by Llama 3.2
Type 'exit' to stop.



Student:  what is AI?



Teacher:
Let's break it down in simple terms.

Artificial Intelligence, or AI, is a way to make machines think and learn like humans. It's a type of computer science that helps machines make decisions, recognize patterns, and even communicate with people. Think of it like a super-smart robot that can perform tasks on its own.

------------------------------------------------------------


Student:  exit



Teacher: Keep learning! 👋


### Stage2 -  input Quries- Generate

In [2]:
import pandas as pd
import ollama

# FILE CONFIGURATION

INPUT_CSV = "ai_teacher_20_questions.csv"
OUTPUT_CSV = "ai_teacher_results.csv"

# LOAD DATASET

df = pd.read_csv(INPUT_CSV)

print("Dataset loaded")
print(f"Number of questions: {len(df)}")

# AI TEACHER INSTRUCTIONS

TEACHER_INSTRUCTIONS = """
You are an expert AI Teacher.

Your job is to teach students, not just provide answers.

Teaching rules:
Give only simple explanation of the concept with layman terms.
Answer should be precise 3 lines.
Response format:
Explain the concept simply.
"""

# GENERATE ANSWER

def generate_answer(question):
    response = ollama.chat(model = "llama3.2", messages=[
            {"role": "system", "content": TEACHER_INSTRUCTIONS},
            {"role": "user", "content": str(question)}
        ]
    )
    return response["message"]["content"].strip()

# GENERATE ANSWERS FOR ALL QUESTIONS    

generated_answers= []

for index, row in df.iterrows():
    question = row["User Query"]

    print(f"Processing {index + 1}/{len(df)}")
    print(f"Question: {question}")

    try:
        answer = generate_answer(question)

    except Exception as e:
        answer = f"ERROR: {str(e)}"

    generated_answers.append(answer)

# APPEND GENERATED ANSWER TO DATAFRAME

df["GeneratedAnswer"] = generated_answers

# SELECT FINAL COLUMNS

df = df[["User Query", "Teacher Response", "GeneratedAnswer"]]

# SAVE RESULT

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

# DISPLAY RESULT
print("\n" + "=" * 70)
print("AI TEACHER TEST COMPLETED")
print("=" * 70)

print(df.to_string(index=False))

print("\nResult saved to:")
print(OUTPUT_CSV)

Dataset loaded
Number of questions: 20
Processing 1/20
Question: What is AI?
Processing 2/20
Question: How does AI differ from traditional programming?
Processing 3/20
Question: What is Machine Learning?
Processing 4/20
Question: What is Deep Learning?
Processing 5/20
Question: What is the difference between AI, ML, and Deep Learning?
Processing 6/20
Question: What is training data?
Processing 7/20
Question: What is a Machine Learning model?
Processing 8/20
Question: What is supervised learning?
Processing 9/20
Question: What is unsupervised learning?
Processing 10/20
Question: What is reinforcement learning?
Processing 11/20
Question: What is an LLM?
Processing 12/20
Question: What is Generative AI?
Processing 13/20
Question: What is a prompt?
Processing 14/20
Question: What is Prompt Engineering?
Processing 15/20
Question: What is RAG?
Processing 16/20
Question: Why is RAG useful?
Processing 17/20
Question: What is a vector database?
Processing 18/20
Question: What are embeddings?
Pr

### Stage3 - LLM Evaluation- Answer Quality

In [3]:
import pandas as pd
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric, GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.models import OllamaModel
from deepeval.evaluate import AsyncConfig

ollama_model = OllamaModel(model="llama3.2")

# LOAD DATA
df = pd.read_csv("ai_teacher_results.csv")

# CREATE TEST CASES
test_cases = [LLMTestCase(input=row["User Query"], actual_output = row["GeneratedAnswer"],
                          expected_output = row["Teacher Response"])
             for _, row in df.iterrows()]

# DEFINE METRICS(ANSWER CORRECTNESS)
correctness = GEval(
    name = "Answer Correctness", model = ollama_model,
    criteria="""
    Evaluate whether the generated answer is factually correct
    compared with the expected teacher answer.
                
    Give a high score when the important meaning and facts are correct.
                
    Penalize:
    - incorrect facts
    - contradictory information
    - missing important information
                
    Minor wording differences should not reduce the score.
    """,
    
    evaluation_params = [SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
    async_mode = AsyncConfig(run_async=False))

relevancy = AnswerRelevancyMetric(model = ollama_model, include_reason = False, async_mode=False)

helpfulness = GEval(
    name = "Helpfulness", model = ollama_model,
    criteria="""
    Evaluate how helpful the generated answer is for a beginner student.

    A helpful answer should:

    1. Directly answer the student's question.
    2. Be easy to understand.
    3. Provide enough explanation.
    4. Avoid unnecessary information.
    5. Help the student understand the concept.
    """,    
    evaluation_params = [SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    async_mode = AsyncConfig(run_async=False))

# RUN DEEPEVAL
result = evaluate(
    test_cases = test_cases,
    metrics=[correctness, relevancy, helpfulness],

    #
    async_config = AsyncConfig(run_async=False))             

✨ You're running DeepEval's latest Answer Correctness [GEval] Metric! (using llama3.2 (Ollama), strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using llama3.2 (Ollama), strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Helpfulness [GEval] Metric! (using llama3.2 (Ollama), strict=False, 
async_mode=False)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_3                                                                                                 │
│  ├──   Input:              What is Deep Learning?                                                               │
│  │     Actual Output:      Deep learning is a type of artificial intelligence that uses many layers of          │
│  │                         neural networks to analyze and learn from data. It's inspired by the way our         │
│  │                         brains process information, with multiple layers of connections that help us         │
│  │                         understand and recognize patterns. By mimicking this brain-like structure, deep      │
│  │                         learning can help machines learn and improve on their own.                           │
│  │     Expected Output:    Deep Learning is a type of Machine Learning that uses neural networks with           │
│  │                         multiple layers to learn complex patterns from large amounts of data.                │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric                     ┃ Score ┃ Threshold ┃ Reason                                          │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Answer Correctness [GEval] │ 0.40  │ 0.50      │ The response contains factual discrepancies,    │
│              │                            │       │           │ including incorrect terminology (Deep           │
│              │                            │       │           │ Learning vs Machine Learning) and missing       │
│              │                            │       │           │ important information (the role of large        │
│              │                            │       │           │ amounts of data).                               │
│        PASS  │ Answer Relevancy           │ 1.00  │ 0.50      │ N/A                                             │
│        PASS  │ Helpfulness [GEval]        │ 0.80  │ 0.50      │ The answer directly addresses the student's     │
│              │                            │       │           │ que...                                          │
│                                                            

⚠ WARNING: No hyperparameters logged.
» ]8;id=1397894;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 1038.36s | token cost: None)
» Test Results (20 total tests):
   » Pass Rate: 70.0% | Passed: 14 | Failed: 6

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [4]:
result.test_results

[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Correctness [GEval]', threshold=0.5, success=True, score=0.6, reason="The response accurately conveys the general meaning of AI, but lacks specificity and technical accuracy, particularly in defining AI as 'the ability of computers to perform tasks that normally require human intelligence'. Minor wording differences do not affect the score.", strict_mode=False, flaky=False, evaluation_model='llama3.2 (Ollama)', error=None, evaluation_cost=0.0, input_tokens=0, output_tokens=0, verbose_logs='Criteria:\n\n    Evaluate whether the generated answer is factually correct\n    compared with the expected teacher answer.\n                \n    Give a high score when the important meaning and facts are correct.\n                \n    Penalize:\n    - incorrect facts\n    - contradictory information\n    - missing important information\n                \n    Minor wording differences should not reduce the score.\n 

In [8]:
for i, test_result in enumerate(result.test_results):
    
    metrics_data = test_result.metrics_data
    
    df.loc[i, "AnswerCorrectness"] = metrics_data[0].score
    df.loc[i, "AnswerRelevancy"] = metrics_data[1].score
    df.loc[i, "Helpfulness"] = metrics_data[2].score

# CALCULATE OVERALL ANSWER QUALITY

df["AnswerQuality"] = (df["AnswerCorrectness"] + df["AnswerRelevancy"] + df["Helpfulness"]) / 3

# ROUND SCORES
metric_columns = ["AnswerCorrectness", "AnswerRelevancy", "Helpfulness", "AnswerQuality"]

df[metric_columns] = df[metric_columns].round(3)

# AVERAGES

average_correctness = df["AnswerCorrectness"].mean()
average_relevancy = df["AnswerRelevancy"].mean()
average_helpfulness = df["Helpfulness"].mean()
average_quality = df["AnswerQuality"].mean()

# QUESTION-WISE RESULTS

print("\n")
print("=" * 120)
print("                 AI TEACHER ANSWER QUALITY")
print("=" * 120)

display_df = df[["User Query", "AnswerCorrectness", "AnswerRelevancy", "Helpfulness", "AnswerQuality"]]

# SAVE FINAL RESULT
df.to_csv("ai_teacher_evaluation.csv", index = False, encoding = "utf-8-sig")

df



                 AI TEACHER ANSWER QUALITY


,User Query,Teacher Response,GeneratedAnswer,AnswerCorrectness,AnswerRelevancy,Helpfulness,AnswerQuality
0,What is AI?,AI (Artificial Intelligence) is the ability of...,AI stands for Artificial Intelligence. It's a ...,0.6,1.000,0.6,0.733
1,How does AI differ from traditional programming?,Traditional programs follow fixed rules writte...,Explain the concept simply:\nTraditional progr...,0.6,1.000,0.8,0.800
2,What is Machine Learning?,Machine Learning is a branch of AI where compu...,Machine Learning is a way for computers to lea...,0.6,1.000,0.9,0.833
3,What is Deep Learning?,Deep Learning is a type of Machine Learning th...,Deep learning is a type of artificial intellig...,0.4,1.000,0.8,0.733
4,"What is the difference between AI, ML, and Dee...",AI is the broad field of making machines intel...,Here's a simple explanation:\n\nAI (Artificial...,0.6,1.000,0.4,0.667
5,What is training data?,Training data is the data used to teach a Mach...,Training data is a set of examples or instance...,0.6,1.000,0.8,0.800
6,What is a Machine Learning model?,A Machine Learning model is a mathematical sys...,A Machine Learning model is a computer program...,0.6,1.000,0.7,0.767
7,What is supervised learning?,Supervised learning trains a model using examp...,Supervised learning is a type of machine learn...,0.6,1.000,0.4,0.667
8,What is unsupervised learning?,Unsupervised learning works with data that doe...,Unsupervised learning is a type of machine lea...,0.6,1.000,0.4,0.667
9,What is reinforcement learning?,Reinforcement learning teaches an agent throug...,Reinforcement learning is a type of machine le...,0.8,1.000,0.8,0.867


In [9]:
average_result = pd.DataFrame(columns=["average_correctness", "average_relevancy", "average_helpfulness", "average_quallity"],
                              data = [[average_correctness, average_relevancy, average_helpfulness, average_quality]])

average_result

,average_correctness,average_relevancy,average_helpfulness,average_quallity
0,0.57,0.98335,0.7,0.75115


In [ ]:
# !pip install deepeval (version-4.3.1)
# pandas version - 2.2.2